# Titanic Survival Prediction

Predicts passenger survival using four tuned models compared against each other — the best performer is selected automatically for the final submission.

## Notebook Structure

| Cell | Description |
|------|-------------|
| 1 | Load `train.csv` and `test.csv` |
| 2 | Impute missing values, encode categorical features |
| 3 | Baseline Random Forest — 80/20 train/val split |
| 4 | Hyperparameter tuning — Random Forest (162 combinations, 5-fold CV) |
| 5 | Hyperparameter tuning — Logistic Regression (18 combinations, 5-fold CV) |
| 6 | Hyperparameter tuning — XGBoost (108 combinations, 5-fold CV) |
| 7 | Hyperparameter tuning — SVM (20 combinations, 5-fold CV) |
| 8 | Preprocess test data, predict with best model, save `submission.csv` |

## Kaggle Results

| # | Model | Params | Features | Public Score |
|---|-------|--------|----------|--------------|
| 1 | SVM | `C=1, kernel=rbf, gamma=scale` | Original 7 | **0.78229** |
| 2 | Random Forest | `max_depth=None, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=300` | Original 7 | **0.77511** |
| 3 | Random Forest | val-score selection | + Title, FamilySize, IsAlone | **0.77511** |
| 4 | XGBoost | `colsample_bytree=0.8, learning_rate=0.1, max_depth=3, n_estimators=100, subsample=1.0` | + Deck, FareBin | **0.76315** |
| 5 | Random Forest | `colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=300, subsample=1.0` | + Title, FamilySize, IsAlone | **0.77511** |
| 6 | XGBoost | `colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=300, subsample=1.0` | + Title, FamilySize, IsAlone, CV selection | **0.77990** |

In [115]:
import pandas as pd

# Load train and test datasets from CSV files
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

# Inspect available feature columns
print(train_data.columns)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')


In [116]:
# Extract title from Name — strong proxy for age, class, and gender
train_data['Title'] = train_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
train_data['Title'] = train_data['Title'].replace(
    ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare'
)
train_data['Title'] = train_data['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# Store median age per title from train — reused for test imputation to prevent leakage
title_age_map = train_data.groupby('Title')['Age'].median()
train_data['Age'] = train_data['Age'].fillna(train_data['Title'].map(title_age_map))
train_data['Title'] = train_data['Title'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4})

train_data['Embarked'] = train_data['Embarked'].fillna(train_data['Embarked'].mode()[0])
train_data['Sex'] = train_data['Sex'].map({'male': 0, 'female': 1})
train_data['Embarked'] = train_data['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})

train_data['FamilySize'] = train_data['SibSp'] + train_data['Parch'] + 1
train_data['IsAlone'] = (train_data['FamilySize'] == 1).astype(int)

In [117]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked',
            'Title', 'FamilySize', 'IsAlone']

X = train_data[features]
y = train_data['Survived']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

accuracy = model.score(X_val, y_val)
print(f"Validation Accuracy: {accuracy}")

Validation Accuracy: 0.8435754189944135


In [118]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Best parameters:    {grid_search.best_params_}")
print(f"Best CV accuracy:   {grid_search.best_score_:.4f}")

model = grid_search.best_estimator_
best_cv_score = grid_search.best_score_
tuned_accuracy = model.score(X_val, y_val)
print(f"Tuned val accuracy: {tuned_accuracy:.4f}  (baseline: {accuracy:.4f})")

Fitting 5 folds for each of 162 candidates, totalling 810 fits
Best parameters:    {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 100}
Best CV accuracy:   0.8272
Tuned val accuracy: 0.8547  (baseline: 0.8436)


In [119]:
from sklearn.linear_model import LogisticRegression

param_grid_lr = [
    {'C': [0.001, 0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']},
    {'C': [0.001, 0.01, 0.1, 1, 10, 100], 'penalty': ['l2'], 'solver': ['lbfgs']}
]

grid_search_lr = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000),
    param_grid_lr,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_lr.fit(X_train, y_train)

print(f"Best parameters:    {grid_search_lr.best_params_}")
print(f"Best CV accuracy:   {grid_search_lr.best_score_:.4f}")

lr_model = grid_search_lr.best_estimator_
lr_cv_score = grid_search_lr.best_score_

print(f"\nLR CV accuracy:     {lr_cv_score:.4f}")
print(f"RF CV accuracy:     {best_cv_score:.4f}")

if lr_cv_score > best_cv_score:
    model = lr_model
    best_cv_score = lr_cv_score
    print("\nSelected model: Logistic Regression")
else:
    print("\nSelected model: Random Forest")

Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best parameters:    {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}
Best CV accuracy:   0.8230

LR CV accuracy:     0.8230
RF CV accuracy:     0.8272

Selected model: Random Forest


c:\Users\BenCliffe\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [120]:
from xgboost import XGBClassifier

param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_search_xgb = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_grid_xgb,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_xgb.fit(X_train, y_train)

print(f"Best parameters:    {grid_search_xgb.best_params_}")
print(f"Best CV accuracy:   {grid_search_xgb.best_score_:.4f}")

xgb_model = grid_search_xgb.best_estimator_
xgb_cv_score = grid_search_xgb.best_score_

print(f"\nXGB CV accuracy:    {xgb_cv_score:.4f}")
print(f"Current best CV:    {best_cv_score:.4f}")

if xgb_cv_score > best_cv_score:
    model = xgb_model
    best_cv_score = xgb_cv_score
    print("\nSelected model: XGBoost")
else:
    print(f"\nPrevious best model retained")

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters:    {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 300, 'subsample': 1.0}
Best CV accuracy:   0.8356

XGB CV accuracy:    0.8356
Current best CV:    0.8272

Selected model: XGBoost


In [121]:
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(random_state=42, class_weight='balanced'))
])

# 5 combinations (was 12) — centred around the previously winning C=1, rbf, gamma=scale
param_grid_svm = [
    {'svc__C': [1, 10], 'svc__kernel': ['linear']},
    {'svc__C': [0.1, 1, 10], 'svc__kernel': ['rbf'], 'svc__gamma': ['scale']}
]

grid_search_svm = GridSearchCV(
    pipeline_svm,
    param_grid_svm,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_svm.fit(X_train, y_train)

print(f"Best parameters:    {grid_search_svm.best_params_}")
print(f"Best CV accuracy:   {grid_search_svm.best_score_:.4f}")

svm_model = grid_search_svm.best_estimator_
svm_cv_score = grid_search_svm.best_score_

print(f"\nSVM CV accuracy:    {svm_cv_score:.4f}")
print(f"Current best CV:    {best_cv_score:.4f}")

if svm_cv_score > best_cv_score:
    model = svm_model
    best_cv_score = svm_cv_score
    print("\nSelected model: SVM")
else:
    print("\nPrevious best model retained")

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best parameters:    {'svc__C': 1, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}
Best CV accuracy:   0.8286

SVM CV accuracy:    0.8286
Current best CV:    0.8356

Previous best model retained


In [122]:
# Apply the same preprocessing to test data before predicting
test_data['Title'] = test_data['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test_data['Title'] = test_data['Title'].replace(
    ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare'
)
test_data['Title'] = test_data['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# Use train-derived title_age_map to prevent leakage
test_data['Age'] = test_data['Age'].fillna(test_data['Title'].map(title_age_map))
test_data['Age'] = test_data['Age'].fillna(title_age_map.median())
test_data['Title'] = test_data['Title'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4})

test_data['Fare'] = test_data['Fare'].fillna(test_data['Fare'].median())
test_data['Embarked'] = test_data['Embarked'].fillna(test_data['Embarked'].mode()[0])
test_data['Sex'] = test_data['Sex'].map({'male': 0, 'female': 1})
test_data['Embarked'] = test_data['Embarked'].map({'C': 0, 'Q': 1, 'S': 2})

test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch'] + 1
test_data['IsAlone'] = (test_data['FamilySize'] == 1).astype(int)

X_test = test_data[features].fillna(test_data[features].median())

predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Submission file created!")

Submission file created!
